# 02 DEM Terrain and Hydrology

This notebook derives terrain and hydrological indicators from the DEM. These layers form the physical basis of the flood susceptibility model.

## What you will do

1. Confirm the preprocessed DEM exists.
2. Understand each DEM derivative.
3. Run the WhiteboxTools hydrology workflow.
4. Preview generated rasters.
5. Decide whether the stream threshold needs tuning.

## Step 1: Load configuration and check DEM input

The hydrology workflow expects a reprojected DEM at `data/interim/reprojected/dem.tif`. If this file is missing, return to notebook 01 and run preprocessing after placing a DEM in `data/raw/dem/`.

In [ ]:
from pathlib import Path
from floodsense.config import load_config

config = load_config()
paths = config['paths']
dem_path = Path(paths['interim_reprojected']) / 'dem.tif'
print('DEM path:', dem_path)
print('DEM exists:', dem_path.exists())

## Step 2: Understand the hydrology outputs

The workflow generates:

- `dem_filled.tif`: DEM with depressions filled.
- `slope.tif`: terrain steepness.
- `flow_direction.tif`: direction of surface-water movement.
- `flow_accumulation.tif`: where water is likely to collect.
- `streams.tif`: stream/drainage proxy extracted from flow accumulation.
- `distance_to_stream.tif`: distance from each cell to stream/drainage proxy.

Flood risk usually increases where elevation is low, slope is gentle, flow accumulation is high, and distance to stream is short.

## Step 3: Check WhiteboxTools availability

Hydrology uses WhiteboxTools. If it is missing, install it with `pip install whitebox` inside your project environment.

In [ ]:
from floodsense.hydrology import check_whitebox_available

whitebox_ready = check_whitebox_available()
print('WhiteboxTools ready:', whitebox_ready)

## Step 4: Run hydrology workflow

This step can take some time for large DEMs. The stream extraction threshold is controlled in `config/config.yaml` under `hydrology.stream_threshold`.

In [ ]:
from floodsense.hydrology import run_hydrology_workflow

if dem_path.exists() and whitebox_ready:
    run_hydrology_workflow(config)
else:
    print('Skipped hydrology. Check DEM availability and WhiteboxTools installation.')

## Step 5: Inspect generated hydrology rasters

In [ ]:
from floodsense.paths import list_input_files

processed_rasters = Path(paths['processed_rasters'])
for file in list_input_files(processed_rasters, ['.tif', '.tiff']):
    print(file.name)

## Step 6: Create preview maps

Preview maps are quick visual checks. They are not final cartographic outputs, but they help you detect obvious processing issues.

In [ ]:
from floodsense.mapping import plot_raster_preview

maps_dir = Path(paths['outputs_maps'])
for name in ['dem_filled', 'slope', 'flow_accumulation', 'streams', 'distance_to_stream']:
    raster = processed_rasters / f'{name}.tif'
    if raster.exists():
        plot_raster_preview(raster, maps_dir / f'{name}_notebook_preview.png', name.replace('_', ' ').title())
        print('Preview saved for', name)
    else:
        print('Missing', raster.name)

## What to check before moving on

- Does `flow_accumulation.tif` show realistic drainage concentration?
- Does `streams.tif` look too dense or too sparse?
- If streams look wrong, adjust `hydrology.stream_threshold` in `config/config.yaml` and rerun this notebook.

Next notebook: `03_flood_susceptibility_model.ipynb`.